# Laboratorio 7 - Quantizacao em Imagens

Este notebook implementa o Laboratorio 7 de DSP, analisando o efeito da quantizacao uniforme em imagens em escala de cinza.

A ideia e reduzir a quantidade de bits por pixel de uma imagem originalmente representada com 8 bits, observar a perda visual e comparar os erros gerados pela reducao de niveis.

## Formula da quantizacao uniforme

Para uma imagem ou sinal de entrada $X$, com $N$ bits, o numero de niveis de quantizacao e:

$$L = 2^N$$

O passo de quantizacao usado no laboratorio e:

$$\Delta = \frac{\max(X) - \min(X)}{2^N - 1}$$

A quantizacao uniforme pode ser feita por:

$$X_q = \operatorname{round}\left(\frac{X - \min(X)}{\Delta}\right)\Delta + \min(X)$$

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from skimage import data
from skimage.color import rgb2gray


plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["image.cmap"] = "gray"


def to_gray_8bit(image: np.ndarray) -> np.ndarray:
    """Converte uma imagem RGB ou em tons de cinza para escala 0..255."""
    image_float = image.astype(float)

    if image_float.ndim == 3:
        image_float = rgb2gray(image_float)

    if image_float.max() <= 1.0:
        image_float = image_float * 255.0

    return np.round(image_float).astype(float)


def quantize_uniform(x: np.ndarray, bits: int) -> np.ndarray:
    """Quantiza uniformemente um vetor ou matriz para 2**bits niveis."""
    if bits < 1:
        raise ValueError("bits precisa ser maior ou igual a 1")

    x = np.asarray(x, dtype=float)
    x_min = np.min(x)
    x_max = np.max(x)

    if np.isclose(x_max, x_min):
        return np.full_like(x, x_min, dtype=float)

    levels = 2**bits
    delta = (x_max - x_min) / (levels - 1)
    xq = np.round((x - x_min) / delta) * delta + x_min
    return np.clip(xq, x_min, x_max)


def quantization_metrics(original: np.ndarray, quantized: np.ndarray) -> dict[str, float]:
    error = original - quantized
    mse = float(np.mean(error**2))
    max_error = float(np.max(np.abs(error)))
    psnr = float("inf") if mse == 0 else float(20 * np.log10(255.0 / np.sqrt(mse)))
    return {"MSE": mse, "erro_maximo": max_error, "PSNR_dB": psnr}


def show_quantization_grid(image: np.ndarray, title: str) -> None:
    bits_cases = [8, 7, 6, 5, 4, 3, 2, 1]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7), constrained_layout=True)
    fig.suptitle(title, fontsize=14)

    for ax, bits in zip(axes.ravel(), bits_cases):
        shown = image if bits == 8 else quantize_uniform(image, bits)
        ax.imshow(shown, vmin=0, vmax=255)
        ax.set_title(f"{bits} bits/pixel")
        ax.axis("off")

    plt.show()


def print_metric_table(image: np.ndarray, bits_cases: list[int] | None = None) -> None:
    if bits_cases is None:
        bits_cases = [7, 6, 5, 4, 3, 2, 1]

    print(f"{'bits':>4} | {'niveis':>6} | {'MSE':>10} | {'erro max':>9} | {'PSNR (dB)':>9}")
    print("-" * 52)
    for bits in bits_cases:
        q = quantize_uniform(image, bits)
        metrics = quantization_metrics(image, q)
        print(
            f"{bits:>4} | {2**bits:>6} | "
            f"{metrics['MSE']:>10.3f} | "
            f"{metrics['erro_maximo']:>9.3f} | "
            f"{metrics['PSNR_dB']:>9.3f}"
        )


## Imagem original

O enunciado sugere usar uma imagem colorida e converter para escala de cinza. Aqui foi usada a imagem `astronaut`, disponivel no pacote `skimage.data`, seguindo a proposta do laboratorio.

In [ ]:
imagem = data.astronaut()
im_gray = to_gray_8bit(imagem)

plt.figure(figsize=(6, 6))
plt.imshow(im_gray, vmin=0, vmax=255)
plt.title("Imagem original com 8 bits por pixel")
plt.axis("off")
plt.show()

print(f"dimensoes: {im_gray.shape}")
print(f"menor nivel: {im_gray.min():.0f}")
print(f"maior nivel: {im_gray.max():.0f}")


## 4) Quantizacao da imagem para 7, 6, 5, 4, 3, 2 e 1 bits/pixel

A funcao `quantize_uniform` recebe uma matriz ou vetor e retorna os valores quantizados em $2^N$ niveis discretos.

In [ ]:
show_quantization_grid(im_gray, "Efeito da quantizacao uniforme na imagem astronaut")

A tabela abaixo mostra o crescimento do erro ao diminuir o numero de bits. O `MSE` mede o erro quadratico medio e o `PSNR` mede a relacao entre o sinal original e o erro de quantizacao. Quanto maior o `PSNR`, mais parecida a imagem quantizada esta da original.

In [ ]:
print_metric_table(im_gray)

## 5) Erros que surgem com a diminuicao do numero de bits

Com menos bits por pixel, a imagem passa a ter menos niveis possiveis de intensidade. Isso aumenta o passo de quantizacao $\Delta$ e faz varios tons proximos serem representados pelo mesmo valor.

Os principais erros observados sao:

- perda de detalhes finos, principalmente em regioes com textura;
- aparecimento de faixas ou blocos de intensidade, tambem conhecido como falso contorno;
- reducao de contraste local em regioes suaves;
- aumento do erro numerico em relacao a imagem original.

Em 7, 6 e 5 bits, a imagem ainda fica visualmente bem proxima da original. Em 4 bits, a reducao de tons ja comeca a aparecer em algumas regioes. Em 3 bits, as faixas ficam evidentes. Em 2 e 1 bits, a imagem fica fortemente degradada.

## 6) Numero de bits em que a imagem deteriora notoriamente

Para a imagem `astronaut`, a deterioracao visual se torna notoria a partir de **3 bits/pixel**. Nesse ponto existem apenas $2^3 = 8$ niveis de cinza, o que torna os contornos artificiais bem visiveis.

Com **4 bits/pixel**, ainda ha $16$ niveis e a imagem permanece reconhecivel, mas ja perde suavidade. Com **2 bits/pixel**, restam apenas $4$ niveis, gerando uma imagem muito simplificada. Com **1 bit/pixel**, restam somente preto e branco, entao praticamente toda a informacao tonal e perdida.

## 7) Teste com diferentes imagens

Para comparar o efeito da quantizacao em outras imagens, foram usadas tambem imagens de exemplo do `skimage.data`. Imagens com regioes suaves tendem a mostrar falso contorno mais cedo. Imagens com muitas bordas e texturas continuam reconheciveis por mais tempo, mas perdem detalhes finos.

In [ ]:
imagens_teste = {
    "camera": to_gray_8bit(data.camera()),
    "coins": to_gray_8bit(data.coins()),
    "coffee": to_gray_8bit(data.coffee()),
}

bits_comparacao = [8, 4, 3, 2, 1]

for nome, img in imagens_teste.items():
    fig, axes = plt.subplots(1, len(bits_comparacao), figsize=(14, 3), constrained_layout=True)
    fig.suptitle(f"Imagem {nome}: comparacao entre niveis de quantizacao", fontsize=13)

    for ax, bits in zip(axes, bits_comparacao):
        shown = img if bits == 8 else quantize_uniform(img, bits)
        ax.imshow(shown, vmin=0, vmax=255)
        ax.set_title(f"{bits} bits")
        ax.axis("off")

    plt.show()

    print(f"Metricas da imagem {nome}")
    print_metric_table(img, bits_cases=[4, 3, 2, 1])
    print()


## Conclusao

A quantizacao reduz a quantidade de valores disponiveis para representar a intensidade dos pixels. Quanto menor o numero de bits, maior o passo $\Delta$ e maior o erro de arredondamento.

Nas imagens testadas, a diferenca em relacao a original e pequena com 7, 6 e 5 bits/pixel. A perda comeca a ficar perceptivel em 4 bits/pixel e se torna claramente notoria em 3 bits/pixel. Em 2 e 1 bits/pixel, a representacao deixa de preservar boa parte das tonalidades da imagem original.